# 01 - Data Cleaning and Preparation

**Kenya Food Price Inflation Tracker**

## Objective
Load and clean the WFP Kenya food price dataset, preparing it for analysis and modeling.

## Steps
1. Load raw WFP Kenya food price data
2. Handle missing values and outliers
3. Standardize date formats and commodity names
4. Filter to focus commodities (Maize, Beans, Rice, Oil, Sugar, Milk)
5. Create cleaned datasets for downstream analysis

## Outputs
- `data/clean/wfp_kenya_clean.csv` - Full cleaned dataset
- `data/clean/wfp_core_staples.csv` - Focus commodities only
- `data/clean/wfp_monthly_avg.csv` - Monthly aggregated prices

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Libraries imported successfully")

## 1. Load Raw Data

In [ ]:
# Load WFP Kenya food prices
df = pd.read_csv('../data/raw/wfp_food_prices_kenya_full.csv')

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['mp_year'].min()}-{df['mp_month'].min():02d} to {df['mp_year'].max()}-{df['mp_month'].max():02d}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Check data types and missing values
print("\n=== Data Info ===")
print(df.info())
print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing[missing > 0]

## 2. Data Cleaning

In [ ]:
# Create proper date column
df['date'] = pd.to_datetime(df[['mp_year', 'mp_month']].assign(day=1))

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

print(f"✓ Created date column")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df[['date', 'mp_commodityname', 'mp_price', 'adm1_name', 'mkt_name']].head(10)

In [ ]:
# Handle missing prices - drop rows with missing prices
print(f"Rows before dropping missing prices: {len(df)}")
df = df.dropna(subset=['mp_price'])
print(f"Rows after dropping missing prices: {len(df)}")

# Remove zero and negative prices (data errors)
print(f"\nRows with price <= 0: {(df['mp_price'] <= 0).sum()}")
df = df[df['mp_price'] > 0]
print(f"Rows after removing invalid prices: {len(df)}")

In [ ]:
# Detect and handle outliers using IQR method (per commodity)
def remove_outliers_iqr(group, column='mp_price', multiplier=3.0):
    """Remove outliers using IQR method"""
    Q1 = group[column].quantile(0.25)
    Q3 = group[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return group[(group[column] >= lower) & (group[column] <= upper)]

print(f"Rows before outlier removal: {len(df)}")
df_clean = df.groupby('mp_commodityname', group_keys=False).apply(remove_outliers_iqr)
print(f"Rows after outlier removal: {len(df_clean)}")
print(f"Removed {len(df) - len(df_clean)} outlier records ({100*(len(df) - len(df_clean))/len(df):.1f}%)")

## 3. Focus on Core Staple Commodities

In [ ]:
# Define core staple commodities
core_staples = [
    'Maize (white) - Retail',
    'Maize - Wholesale',
    'Beans (dry) - Retail',
    'Beans - Wholesale',
    'Rice - Retail',
    'Oil (vegetable) - Retail',
    'Sugar - Retail',
    'Milk (cow, pasteurized) - Retail'
]

# Filter to core staples
df_staples = df_clean[df_clean['mp_commodityname'].isin(core_staples)].copy()

print(f"Core staples dataset: {df_staples.shape}")
print(f"\nCommodity distribution:")
print(df_staples['mp_commodityname'].value_counts())

## 4. Create Monthly Aggregated Data

In [ ]:
# Aggregate to monthly averages (by commodity and market)
df_monthly = df_staples.groupby(
    ['date', 'mp_commodityname', 'adm1_name', 'mkt_name']
).agg({
    'mp_price': ['mean', 'std', 'count'],
    'cm_name': 'first',
    'mkt_id': 'first',
    'latitude': 'first',
    'longitude': 'first'
}).reset_index()

# Flatten column names
df_monthly.columns = ['date', 'commodity', 'region', 'market', 
                       'price_mean', 'price_std', 'obs_count',
                       'currency', 'market_id', 'latitude', 'longitude']

print(f"Monthly aggregated dataset: {df_monthly.shape}")
df_monthly.head(10)

## 5. Save Cleaned Datasets

In [ ]:
# Create clean data directory if it doesn't exist
Path('../data/clean').mkdir(parents=True, exist_ok=True)

# Save cleaned datasets
df_clean.to_csv('../data/clean/wfp_kenya_clean.csv', index=False)
print(f"✓ Saved: wfp_kenya_clean.csv ({len(df_clean)} rows)")

df_staples.to_csv('../data/clean/wfp_core_staples.csv', index=False)
print(f"✓ Saved: wfp_core_staples.csv ({len(df_staples)} rows)")

df_monthly.to_csv('../data/clean/wfp_monthly_avg.csv', index=False)
print(f"✓ Saved: wfp_monthly_avg.csv ({len(df_monthly)} rows)")

## 6. Data Quality Summary

In [ ]:
print("=== DATA CLEANING SUMMARY ===")
print(f"\nOriginal dataset: {df.shape[0]:,} rows")
print(f"After cleaning: {df_clean.shape[0]:,} rows")
print(f"Records removed: {df.shape[0] - df_clean.shape[0]:,} ({100*(df.shape[0] - df_clean.shape[0])/df.shape[0]:.1f}%)")
print(f"\nCore staples dataset: {df_staples.shape[0]:,} rows")
print(f"Date range: {df_staples['date'].min()} to {df_staples['date'].max()}")
print(f"Unique commodities: {df_staples['mp_commodityname'].nunique()}")
print(f"Unique markets: {df_staples['mkt_name'].nunique()}")
print(f"Unique regions: {df_staples['adm1_name'].nunique()}")

print("\n✓ Data cleaning completed successfully!")